In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as F
from sklearn.preprocessing import MinMaxScaler
import math

device = torch.device('cuda'if torch.cuda.is_available() else "cpu")

inputs = pd.read_excel("old/data/input_parameters.xlsx")
outputs_re = pd.read_excel("old/data/reel.xlsx")
outputs_im = pd.read_excel("old/data/imaginary.xlsx")


In [2]:
inputs.columns

Index(['length of patch', 'width of patch', 'height of patch',
       'height of substrate', 'height of solder resist layer',
       'radius of the probe', 'c_pad', 'c_antipad', 'c_probe',
       'dielectric constant of substrate',
       'dielectric constant of solder resist layer'],
      dtype='object')

In [3]:

# Apply the FFT
s_param = outputs_re + 1j*outputs_im
s_param_fft = np.fft.fft(s_param, axis=-1)
s_fft_re = np.real(s_param_fft)
s_fft_im = np.imag(s_param_fft)

# Normalize the inputs
input_scaler = MinMaxScaler((-1, 1))
normalized_inputs = input_scaler.fit_transform(inputs)
# Create three dimensional output 
outputs = np.array([[outputs_re], [outputs_im]]) 
normalized_outputs = outputs.reshape(2, int(len(inputs)), 201)

# Separating data as train and test
X_train = []
X_test = []
y_train = []
y_test = []
s_fft_train_re = []
s_fft_test_re = []
s_fft_train_im = []
s_fft_test_im = []

for x in range(5000):
    if x % 5 == 0:
        X_test.append(normalized_inputs[x, :])
        y_test.append(normalized_outputs[:, x, :])   
        s_fft_test_re.append(s_fft_re[x,:])
        s_fft_test_im.append(s_fft_im[x,:])
        continue

    X_train.append(normalized_inputs[x, :])
    y_train.append(normalized_outputs[:, x, :]) 
    s_fft_train_re.append(s_fft_re[x,:])
    s_fft_train_im.append(s_fft_im[x,:])

X_train = np.array(X_train)
y_train = np.array(y_train)
s_fft_train_re = np.array(s_fft_train_re)
s_fft_train_im = np.array(s_fft_train_im)
X_test = np.array(X_test)
y_test = np.array(y_test)  
s_fft_test_re = np.array(s_fft_test_re)
s_fft_test_im = np.array(s_fft_test_im)

X_train = torch.from_numpy(X_train.astype(np.float32)).to(device)  
X_test = torch.from_numpy(X_test.astype(np.float32)).to(device)  
y_train = torch.from_numpy(y_train.astype(np.float32)).to(device) 
y_test = torch.from_numpy(y_test.astype(np.float32)).to(device)  
s_fft_train_re = torch.from_numpy(s_fft_train_re.astype(np.float32)).to(device)  
s_fft_test_re = torch.from_numpy(s_fft_test_re.astype(np.float32)).to(device)  
s_fft_train_im = torch.from_numpy(s_fft_train_im.astype(np.float32)).to(device)  
s_fft_test_im = torch.from_numpy(s_fft_test_im.astype(np.float32)).to(device)  

# Neural Network Structure
class addCoords_1D(nn.Module):
   def __init__(self):
       super(addCoords_1D, self).__init__()

   def forward(self, out):
       in_batch, in_ch, in_w = out.shape        
       width_coords = torch.linspace(-1, 1, steps=in_w).to(out.device)   
       wc = width_coords.repeat(in_batch, 1, 1)
       coord_x = torch.cat((out, wc), 1)
       return coord_x
   
class NeuralNet(nn.Module):
    def __init__(self):
        super(NeuralNet, self).__init__()
        self.logsigma = nn.Parameter(torch.ones(1))
        self.filter_size = 41
        
        self.add_coords = addCoords_1D()
        
        self.linear = nn.Sequential(
            nn.Linear(11, 60),                                
            nn.Tanh(), 
            
            nn.Linear(60, 60),            
            nn.Tanh(),  
            
            nn.Linear(60, 60),            
            nn.Tanh(), 
                                   
        )                  
                               
        self.deconv1 = nn.Sequential(
            nn.ConvTranspose1d(in_channels= 60 , out_channels= 40 , kernel_size=21, stride=1),                     
            nn.Tanh(), 
                                             
        )     
        self.deconv2 = nn.Sequential(
            nn.ConvTranspose1d(in_channels= 40 , out_channels=40, kernel_size=7, stride=3),             
            nn.Tanh(),   
                           
        )
        self.deconv3 = nn.Sequential(
            nn.ConvTranspose1d(in_channels=41, out_channels=2, kernel_size=3, stride=3),             
            nn.Tanh(), 
                    
        )                        
         
    def smooth(self, x):
            
             # First define a global variable for the size of the Gaussian filter. 
             # Here, filter_size is one half of the filter. This is a hyperparameter to tune.
             N = self.filter_size
             
            
             # Pad the input tensor to preserve the dimension
             x = F.pad(x, ((N-1)//2, (N-1)//2), mode='reflect')
             
             # Use the natural log of sigma as the variable for scaling.
             sigma = torch.exp(self.logsigma).unsqueeze(-1).unsqueeze(-1) # Tensor size N_channelx1x1
             sigma = sigma.repeat((1, 1, N))# Tensor size N_channelxN_channelxfilter_size
             
             # Set the total width of the Gaussian filter as 6*sigma
             hw = 3*sigma
             # Generate a vector between 0 and 1
             xx = torch.linspace(0,1,steps=N).unsqueeze(0).unsqueeze(0).to(device)# Tensor size 1x1xfilter_size
             xx = xx.repeat((x.shape[1], 1,1))# Tensor size N_channelxN_channelxfilter_size
             # Shift xx for every channel 
             xx = 2*hw*xx-hw # Tensor size N_channelxN_channelxfilter_size
             
             # Generate the Gaussian filter
             gauss = 1/(2*math.pi*sigma**2)*torch.exp(-1/(2*(sigma**2))*xx**2)# Tensor size N_channelxN_channelxfilter_size
             
             # Find the sum of the coeffcients
             gauss_sum = gauss.sum(dim=2).unsqueeze(-1)# Tensor size N_channelxN_channelx1
             gauss_sum = gauss_sum.repeat((1, 1, gauss.shape[-1]))# Tensor size N_channelxN_channelxfilter_size
             # Normalize the filter
             gauss = gauss/gauss_sum # Tensor size N_channelxN_channelxfilter_size
             output = F.conv1d(x, gauss, groups=x.shape[1] )
             
             return output  
         
    def forward(self, x):

        out = self.linear(x)

        out = out.view(len(x), 60 , 1)

        out = self.deconv1(out)
        
        out = self.deconv2(out)
        
        out = self.add_coords(out)
        
        out = self.deconv3(out)  
                              
        out = self.smooth(out)
       
        return out

# Define the model
model = NeuralNet().to(device)    
# Optimizer and parameters
learning_rate = 0.007
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=0.7)
criterion = nn.MSELoss(size_average=None, reduce=None, reduction='mean') 

# Load the weights and optimizer
def load_checkpoint(checkpoint):
    print("---loading checkpoint")
    model.load_state_dict(checkpoint['state_dict'])    
    optimizer.load_state_dict(checkpoint['optimizer'])  

#load_checkpoint(torch.load("initial_weight.pth")) #The initial weight for all runs
#load_checkpoint(torch.load("huber_w_fft.pth")) # Weight for Huber loss with FFT regularization 
#load_checkpoint(torch.load("huber_w_f_fft.pth"))  # Weight for Huber loss with F-FFT regularization 
load_checkpoint(torch.load("huber_w_f_fft_mag.pth", map_location=torch.device('cpu'))) # Weight for Huber loss with F-FFT and magnitude regularization 

# Constants for different implementations
beta = 0.1 # for all
alfa = beta # for FFT
w_huber = 0.1 # for FFT
#alfa = 0.001 # for F-FFT
k = np.concatenate((np.arange(0, 40), np.arange(161,201))) # filter size for F-FFT
w_filter = 0.05 # for F-FFT
#wmag = 1.2 # for F-FFT w/ magnitude
wmag = 0.4 # for huber w/ magnitude


# Loss
def huber_loss(x, y, z, t):
    
    # Huber loss 
    a = torch.where(torch.abs(x - y) < beta, 0.5*torch.square(x - y)/beta, torch.zeros(int(len(y_train)),2, 201).to(device))
    b = torch.where(torch.abs(x - y) >= beta, torch.abs(x - y) - 0.5*beta, torch.zeros(int(len(y_train)),2, 201).to(device)) 
    lossa = torch.mean(a + b, dim=2)
    
    # Calculate FFT of predicted output
    real_pred = x[:,0,:]
    imaginary_pred = x[:,1,:]
    s_pred = real_pred + 1j*imaginary_pred
    s_fft_pred = np.fft.fft(s_pred.detach().numpy(), axis=-1)
    s_fft_pred_re = torch.real(torch.tensor(s_fft_pred)).float() 
    s_fft_pred_im = torch.imag(torch.tensor(s_fft_pred)).float()
    
    # Huber loss for real and imaginary part of FFT
    '''c = torch.where(torch.abs(s_fft_pred_re - z) < alfa, 0.5*torch.square(s_fft_pred_re - z)/alfa, torch.zeros(int(len(y_train)), 201).to(device))
    d = torch.where(torch.abs(s_fft_pred_re - z) >= alfa, torch.abs(s_fft_pred_re - z) - 0.5*alfa, torch.zeros(int(len(y_train)), 201).to(device)) 
    lossa_fft_re = torch.mean((c + d), dim=1)
    
    e = torch.where(torch.abs(s_fft_pred_im - t) < alfa, 0.5*torch.square(s_fft_pred_im - t)/alfa, torch.zeros(int(len(y_train)), 201).to(device))
    f = torch.where(torch.abs(s_fft_pred_im - t) >= alfa, torch.abs(s_fft_pred_im - t) - 0.5*alfa, torch.zeros(int(len(y_train)), 201).to(device)) 
    lossa_fft_im = torch.mean((e + f), dim=1)'''
    
    # Huber loss for real and imaginary part F-FFT
    c = torch.where(torch.abs(s_fft_pred_re - z) <alfa, 0.5*torch.square(s_fft_pred_re - z)/alfa, torch.zeros(int(len(y_train)), 201).to(device))
    d = torch.where(torch.abs(s_fft_pred_re - z) >=alfa, torch.abs(s_fft_pred_re - z) - 0.5*alfa, torch.zeros(int(len(y_train)), 201).to(device)) 
    sum1 = (c+d).to(device)
    sum1 = sum1[:,k].to(device)
    lossa_fft_re = torch.mean((sum1), dim=1)
 
    e = torch.where(torch.abs(s_fft_pred_im - t) < alfa, 0.5*torch.square(s_fft_pred_im - t)/alfa, torch.zeros(int(len(y_train)), 201).to(device))
    f = torch.where(torch.abs(s_fft_pred_im - t) >= alfa, torch.abs(s_fft_pred_im - t) - 0.5*alfa, torch.zeros(int(len(y_train)), 201).to(device)) 
    sum2 = (e+f).to(device)
    sum2 = sum2[:,k].to(device)
    lossa_fft_im = torch.mean((sum2), dim=1)
    
    # Magnitude Regularization
    real_orig = y[:,0,:]
    imaginary_orig = y[:,1,:]
    mag_pred = torch.sqrt(real_pred**2 + imaginary_pred**2)
    mag_orig = torch.sqrt(real_orig**2 + imaginary_orig**2)
    g = torch.where(torch.abs(mag_pred - mag_orig) < alfa, 0.5*torch.square(mag_pred - mag_orig)/alfa, torch.zeros(int(len(y_train)), 201).to(device))
    i = torch.where(torch.abs(mag_pred - mag_orig) >= alfa, torch.abs(mag_pred - mag_orig) - 0.5*alfa, torch.zeros(int(len(y_train)), 201).to(device)) 
    lossa_mag = torch.mean(g + i, dim=1) 
    
    return torch.mean(lossa) + torch.mean(lossa_fft_re + lossa_fft_im)*w_filter + torch.mean(lossa_mag)*wmag  

# Training loop
num_epochs = 3200
batch_size = 1

data_loader_1 = torch.utils.data.DataLoader(X_train, batch_size=int(len(X_train)/batch_size), shuffle = False)
data_loader_2 = torch.utils.data.DataLoader(y_train, batch_size=int(len(y_train)/batch_size), shuffle = False)
   
number_of_epochs = []
tr_huber_data = []
te_mse_data = []

for epoch in range(num_epochs):
    # Save the weights
    '''def save_checkpoint(state, filename='initial_hubertrial_3.pth'):
        print("--saving weight")
        torch.save(state, filename)
    
    if epoch == 3299 :  
        checkpoint = {'state_dict': model.state_dict(), 'optimizer': optimizer.state_dict()}
        save_checkpoint(checkpoint) '''
        
    # Forward and backward pass 
    for batch_idx, (X_train, y_train) in enumerate(zip(data_loader_1,data_loader_2)):
        
        model.train()       
        output_train = model(X_train).to(device)   
        output_train_1 = torch.reshape(output_train, (int(len(X_train)), 2, 201)) # Predicted output
        
        train_loss = huber_loss(output_train_1, y_train, s_fft_train_re, s_fft_train_im) # Calculate loss
        
        output_train_predicted = output_train_1.detach().cpu().numpy() # Predicted output in numpy

        train_loss.backward()
        optimizer.step()
                        
        if batch_idx == 0:
            scheduler.step()
        optimizer.zero_grad()       
        model.eval()
        with torch.no_grad():           
            output_test = model(X_test).to(device)          
            output_test = torch.reshape(output_test, (int(len(y_test)), 2, 201)) # Predicted output
                               
            test_loss = criterion(output_test, y_test)  # Calculate loss (to compare the previous implementations test loss is printed as MSE)
   
            output_test_predicted = output_test.detach().cpu().numpy()  # Predicted output in numpy

            if (epoch) % 1 == 0 :
                print(f'Epoch [{epoch + 1}/{num_epochs}], Batch [{batch_idx+1}/{batch_size}], Tr_Loss:{train_loss.item(): .8f}, '
                                                f'Te_Loss: {test_loss.item():.8f}')
                                               
            if batch_idx == 0:
  
                tr_huber_data = np.append(tr_huber_data, train_loss.detach().cpu()) 
                te_mse_data = np.append(te_mse_data, test_loss.detach().cpu())              
                number_of_epochs = np.append(number_of_epochs, epoch + 1)                     
    pass



---loading checkpoint
Epoch [1/3200], Batch [1/1], Tr_Loss: 0.02134836, Te_Loss: 0.00253250
Epoch [2/3200], Batch [1/1], Tr_Loss: 0.02219597, Te_Loss: 0.00250657
Epoch [3/3200], Batch [1/1], Tr_Loss: 0.02202882, Te_Loss: 0.00247445
Epoch [4/3200], Batch [1/1], Tr_Loss: 0.02161634, Te_Loss: 0.00248790
Epoch [5/3200], Batch [1/1], Tr_Loss: 0.02176183, Te_Loss: 0.00249310
Epoch [6/3200], Batch [1/1], Tr_Loss: 0.02176314, Te_Loss: 0.00248705
Epoch [7/3200], Batch [1/1], Tr_Loss: 0.02153117, Te_Loss: 0.00249045
Epoch [8/3200], Batch [1/1], Tr_Loss: 0.02144337, Te_Loss: 0.00248912
Epoch [9/3200], Batch [1/1], Tr_Loss: 0.02148213, Te_Loss: 0.00247794
Epoch [10/3200], Batch [1/1], Tr_Loss: 0.02150528, Te_Loss: 0.00246479
Epoch [11/3200], Batch [1/1], Tr_Loss: 0.02142630, Te_Loss: 0.00246289
Epoch [12/3200], Batch [1/1], Tr_Loss: 0.02138720, Te_Loss: 0.00247608
Epoch [13/3200], Batch [1/1], Tr_Loss: 0.02150282, Te_Loss: 0.00248067
Epoch [14/3200], Batch [1/1], Tr_Loss: 0.02147490, Te_Loss: 0.00

KeyboardInterrupt: 

In [99]:
import joblib
joblib.dump(input_scaler, 'NNModel/scaler.gz')

['NNModel/scaler.gz']

In [3]:

epoch_samp = num_epochs

# Create predicted output matrices 
te_output_last = np.reshape(output_test_predicted, (int(len(X_test)), 2, 201))
# Reshape to obtain the real and imaginary parts 
c1_test = np.array(te_output_last[:,0,:])
c2_test = np.array(te_output_last[:,1,:])
te2_output_last = np.array([[c1_test], [c2_test]])
te_output_last_denorm = te2_output_last.reshape(2,int(len(X_test)),201)
# Predicted real and imaginary outputs
last_re_pred = te_output_last_denorm[0,:,:]
last_im_pred = te_output_last_denorm[1,:,:]
# Same process for the main data
y_te_main = np.array(y_test.detach().cpu(), dtype=np.float32)
d1_test = np.array(y_te_main[:,0,:])
d2_test = np.array(y_te_main[:,1,:])
y_te_main_denorm = np.array([[d1_test],[d2_test]])
y_te_main_denorm = y_te_main_denorm.reshape(2, int(len(X_test)),201)
last_re_main = y_te_main_denorm[0,:,:]
last_im_main = y_te_main_denorm[1,:,:]
# Create magnitude matrices for both predicted and main data
last_mag_pred = np.sqrt(np.square(last_re_pred) + np.square(last_im_pred))
last_mag_main = np.sqrt(np.square(last_re_main) + np.square(last_im_main))

# Obtain the MSE losses
last_mse_re = np.mean(np.square(last_re_pred - last_re_main))
last_mse_im = np.mean(np.square(last_im_pred - last_im_main))
last_mse_mag = np.mean(np.square(last_mag_pred - last_mag_main))

# Obtain the RSE losses
last_rse_re = np.zeros((int(len(y_test)), 201))
summa_re = np.square((last_re_pred - last_re_main)).sum(axis=1)
summa_main_re = np.square(last_re_main - last_re_main.mean(axis=1).reshape(int(len(y_test)),1)).sum(axis=1)    
last_rse_re = summa_re/summa_main_re

last_rse_im = np.zeros((int(len(y_test)), 201))
summa_im = np.square((last_im_pred - last_im_main)).sum(axis=1)
summa_main_im = np.square(last_im_main - last_im_main.mean(axis=1).reshape(int(len(y_test)),1)).sum(axis=1)    
last_rse_im = summa_im/summa_main_im

last_rse_mag = np.zeros((int(len(y_test)), 201))
summa_mag = np.square((last_mag_pred - last_mag_main)).sum(axis=1)
summa_main_mag = np.square(last_mag_main - last_mag_main.mean(axis=1).reshape(int(len(y_test)),1)).sum(axis=1)    
last_rse_mag = summa_mag/summa_main_mag

# Number of passivity violated samples
a = np.where(last_mag_pred > 1, last_mag_pred, 0)
c = np.argwhere(a>0)
c = c[:,0]
c = np.unique(c)     

# Measure the smoothness
dx = (33-23)/200

dy_re = np.diff(te_output_last_denorm[0,:,:])
dy_im = np.diff(te_output_last_denorm[1,:,:])

dy_re_orig = np.diff(y_te_main_denorm[0,:,:])
dy_im_orig = np.diff(y_te_main_denorm[1,:,:])  

# Predicted real output
re_arclength = np.zeros((1000, 192))
for j in range(192):
    for i in range(10):
        re_arclength[:,j] = np.sum(np.sqrt(1 + (dy_re[:,j:j+i]/dx)**2)*dx, axis=1)

# Main real output        
re_arclength_orig = np.zeros((1000, 192))
for j in range(192):
    for i in range(10):
        re_arclength_orig[:,j] = np.sum(np.sqrt(1 + (dy_re_orig[:,j:j+i]/dx)**2)*dx, axis=1)

# Predicted imaginary output
im_arclength = np.zeros((1000, 192))
for j in range(192):
    for i in range(10):
        im_arclength[:,j] = np.sum(np.sqrt(1 + (dy_im[:,j:j+i]/dx)**2)*dx, axis=1)
 
# Main imaginary output               
im_arclength_orig = np.zeros((1000, 192))
for j in range(192):
    for i in range(10):
        im_arclength_orig[:,j] = np.sum(np.sqrt(1 + (dy_im_orig[:,j:j+i]/dx)**2)*dx, axis=1)
        
# Normalize arc length
re_smo_der = np.abs(re_arclength - re_arclength_orig)/np.abs(re_arclength_orig) 
im_smo_der = np.abs(im_arclength - im_arclength_orig)/np.abs(im_arclength_orig)

sum_re_smo_der = re_smo_der.sum(axis=1)
sum_im_smo_der = im_smo_der.sum(axis=1)

smoothness = (np.mean(sum_im_smo_der) + np.mean(sum_re_smo_der))/2


In [ ]:
torch.save(model.state_dict(), "trained_model.pt")

In [102]:
torch.save(model,"NNModel/NN.pt")